<a href="https://colab.research.google.com/github/IsmoilDev7/Tashkent-Housing-Price-Prediction/blob/main/Tashkent_Housing_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/anvarnarz/praktikum_datasets/main/housing_data_08-02-2021.csv')
df.head()

,location,district,rooms,size,level,max_levels,price
0,"город Ташкент, Юнусабадский район, Юнусабад 8-...",Юнусабадский,3,57,4,4,52000
1,"город Ташкент, Яккасарайский район, 1-й тупик ...",Яккасарайский,2,52,4,5,56000
2,"город Ташкент, Чиланзарский район, Чиланзар 2-...",Чиланзарский,2,42,4,4,37000
3,"город Ташкент, Чиланзарский район, Чиланзар 9-...",Чиланзарский,3,65,1,4,49500
4,"город Ташкент, Чиланзарский район, площадь Актепа",Чиланзарский,3,70,3,5,55000


In [ ]:
# Avval raqamdan boshqa belgilarni olib tashlaymiz (masalan, "m²" va boshqalar)
df['size'] = df['size'].astype(str).str.extract('(\d+)')  # faqat raqamlarni ajratadi
df['size'] = pd.to_numeric(df['size'], errors='coerce')  # str → float (xatoliklar NaN bo'ladi)

# Endi NaN qiymatlarni olib tashlash (agar kerak bo‘lsa)
df = df.dropna(subset=['size'])

# Endi solishtirishni amalga oshirish mumkin
df = df[df['size'] < 250]

# Avval raqamdan boshqa belgilarni olib tashlaymiz (masalan, "m²" va boshqalar)
df['price'] = df['price'].astype(str).str.extract('(\d+)')
df['price'] = pd.to_numeric(df['price'], errors= 'coerce')

# Endi NaN qiymatlarni olib tashlash (agar kerak bo‘lsa)
df = df.dropna(subset = ['price'])

# Endi solishtirishni amalga oshirish mumkin
df = df[df['price'] < 200000]


In [ ]:
df.describe()

,rooms,size,level,max_levels,price
count,7273.000000,7273.000000,7273.000000,7273.000000,7273.000000
mean,2.576378,69.889042,3.691187,5.997113,53936.597415
std,1.028995,28.780192,2.235773,2.598154,29246.140505
min,1.000000,1.000000,1.000000,1.000000,2.000000
25%,2.000000,50.000000,2.000000,4.000000,34600.000000
50%,2.000000,65.000000,3.000000,5.000000,45500.000000
75%,3.000000,83.000000,5.000000,9.000000,65000.000000
max,8.000000,246.000000,19.000000,25.000000,198000.000000


In [ ]:
df['roomssize'] = df['size'] / df['rooms']
df['leveldegree'] = df['max_levels'] / df['level']

#Train va test ajratish

In [ ]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(df, test_size = 0.2, random_state = 42)

X_train = train_set.drop("price", axis = 1)
Y_train = train_set["price"].copy()

X_test = test_set.drop("price", axis = 1)
Y_test = test_set["price"].copy()

#endi raqam va matn qismlarni ajratib olamiz
categorical_cols = X_train['district'].copy()
numeric_cols = X_train.drop(columns = 'district')

#Pipeline ichida yangi ustunlar qoshish uchun classdan foydalanamiz

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
rooms, size, level, max_levels = 0,1,2,3 # df ichida ustun indexsi

class CobbinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_roomsize_leveldegree = True):
        self.add_roomsize_leveldegree = add_roomsize_leveldegree
    def fit(self, X, y = None):
        return self
    def transform(self, X):
        size_per_room = X[:, size] / X[:, rooms]
        level_degree = X[:, level] / X[:, max_levels]
        if self.add_roomsize_leveldegree:
            return np.c_[X, size_per_room, level_degree]
        else:
            return np.c_[X, size_per_room]

In [ ]:
df

,district,rooms,size,level,max_levels,price,roomssize,leveldegree
0,Юнусабадский,3,57,4,4,52000,19.000000,1.000000
1,Яккасарайский,2,52,4,5,56000,26.000000,1.250000
2,Чиланзарский,2,42,4,4,37000,21.000000,1.000000
3,Чиланзарский,3,65,1,4,49500,21.666667,4.000000
4,Чиланзарский,3,70,3,5,55000,23.333333,1.666667
...,...,...,...,...,...,...,...,...
7560,Яшнободский,1,38,5,5,24500,38.000000,1.000000
7561,Яшнободский,2,49,1,4,32000,24.500000,4.000000
7562,Шайхантахурский,2,64,3,9,40000,32.000000,3.000000
7563,Мирзо-Улугбекский,1,18,1,4,11000,18.000000,4.000000


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'median')),
    ('attribs', CobbinedAttributesAdder()),
    ('std_scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy= 'most_frequent')),
    ('cat_encoder', OneHotEncoder(handle_unknown= 'ignore'))
])

full_pipeline = ColumnTransformer([
    ('num', num_pipeline, ['rooms', 'size', 'level', 'max_levels']),
    ('cat', cat_pipeline, ['district'])
])

#datani pipelinedan otqazish orqali modelga tayor holatga keltiramiz

In [ ]:
X_prepared = full_pipeline.fit_transform(X_train)
X_test_prepared = full_pipeline.transform(X_test)

#Modellash

In [ ]:
from sklearn.linear_model import LinearRegression

LR_model = LinearRegression()
LR_model.fit(X_prepared, Y_train)

LinearRegression()

#Tayyor modeli orqali qiymatlarni bashorat qilib koramiz

In [ ]:
test_data = X_test.sample(5)
test_label = Y_test[test_data.index]
test_data_prepared = full_pipeline.transform(test_data)

In [ ]:
predicted_data = LR_model.predict(test_data_prepared)
pd.DataFrame({'Prognoz': predicted_data, 'Real': test_label})

,Prognoz,Real
6595,34448.670177,69000
1683,33937.905676,25907
341,51422.336754,44000
2732,29138.476586,22000
6350,50414.866196,48000
